In [1]:
import os
import numpy as np
import nibabel as nib

In [2]:
def load_nii(path):
    nii = nib.load(path)
    return nii.get_fdata()

In [1]:
def clip_hu(volume, min_hu=-200, max_hu=250):
    return np.clip(volume, min_hu, max_hu)

def normalize(volume):
    mean = np.mean(volume)
    std = np.std(volume)
    return (volume - mean) / (std + 1e-8)

def clean_mask(mask):
    return mask.astype(np.uint8)

def remove_empty_slices(volume, mask):

    v_slices = []
    m_slices = []

    # volume shape = (D, H, W)
    for i in range(volume.shape[0]):
        if np.sum(mask[i]) > 0:
            v_slices.append(volume[i])
            m_slices.append(mask[i])

    return np.array(v_slices), np.array(m_slices)

def extract_3d_patches(volume, mask,
                       patch_size=(16, 128, 128),
                       stride=16):

    D, H, W = volume.shape
    d, h, w = patch_size

    vol_patches = []
    mask_patches = []

    for z in range(0, D - d + 1, stride):
        for y in range(0, H - h + 1, stride):
            for x in range(0, W - w + 1, stride):

                v_patch = volume[z:z+d, y:y+h, x:x+w]
                m_patch = mask[z:z+d, y:y+h, x:x+w]

                if np.sum(m_patch) > 0:
                    vol_patches.append(v_patch)
                    mask_patches.append(m_patch)

    return vol_patches, mask_patches

In [2]:
def preprocess_image(image_path, mask_path):

    image = load_nii(image_path)
    mask = load_nii(mask_path)

    image = np.transpose(image, (2, 0, 1))
    mask = np.transpose(mask, (2, 0, 1))

    image = clip_hu(image)
    image = normalize(image)
    mask = clean_mask(mask)

    image, mask = remove_empty_slices(image, mask)

    if image.ndim != 3 or image.shape[0] < 16:
        return [], []

    img_patches, mask_patches = extract_3d_patches(image, mask)

    return img_patches, mask_patches

In [3]:
def load_dataset(image_dir, mask_dir):

    image_files = sorted(os.listdir(image_dir))

    X = []
    Y = []

    for file in image_files:

        if not file.endswith(".nii") and not file.endswith(".nii.gz"):
            continue

        image_path = os.path.join(image_dir, file)

        mask_file = file.replace("volume", "segmentation")
        mask_path = os.path.join(mask_dir, mask_file)

        if not os.path.exists(mask_path):
            print("Mask not found for:", file)
            continue

        img_patches, mask_patches = preprocess_image(image_path, mask_path)

        X.extend(img_patches)
        Y.extend(mask_patches)

    return np.array(X), np.array(Y)


In [6]:
image_dir = r"C:\\MiniProject\\Mini Project\\LiTS_small\\images"
mask_dir = r"C:\\MiniProject\\Mini Project\\LiTS_small\\masks"

X, Y = load_dataset(image_dir, mask_dir)

print("\nFinal dataset shape:")
print("Images:", X.shape)
print("Masks :", Y.shape)

MemoryError: Unable to allocate 51.6 GiB for an array with shape (26403, 16, 128, 128) and data type float64

In [4]:
import os
import numpy as np
import nibabel as nib


# ===============================
# Load NII File
# ===============================

def load_nii(path):
    nii = nib.load(path)
    data = np.array(nii.dataobj, dtype=np.float32)
    return data
# ===============================
# Preprocessing
# ===============================

def clip_hu(volume, min_hu=-200, max_hu=250):
    return np.clip(volume, min_hu, max_hu)


def normalize(volume):

    volume = volume.astype(np.float32)

    mean = volume.mean()
    std = volume.std()

    volume -= mean
    volume /= (std + 1e-8)

    return volume

def clean_mask(mask):
    return mask.astype(np.uint8)   # multi-class safe


# ===============================
# Remove Empty Slices
# ===============================

def remove_empty_slices(volume, mask):

    v_slices = []
    m_slices = []

    for i in range(volume.shape[0]):
        if np.sum(mask[i]) > 0:
            v_slices.append(volume[i])
            m_slices.append(mask[i])

    return np.array(v_slices), np.array(m_slices)


# ===============================
# Full Volume Processing (NO PATCHES)
# ===============================

def preprocess_image(image_path, mask_path):

    image = load_nii(image_path)
    mask = load_nii(mask_path)

    # LiTS orientation fix
    image = np.transpose(image, (2, 0, 1))
    mask = np.transpose(mask, (2, 0, 1))

    image = clip_hu(image).astype(np.float32)
    image = normalize(image)
    mask = clean_mask(mask)

    image, mask = remove_empty_slices(image, mask)

    if image.ndim != 3:
        return None, None

    return image.astype(np.float32), mask.astype(np.uint8)


# ===============================
# Load Dataset (Volume-wise)
# ===============================   
def load_dataset(image_dir, mask_dir):

    image_files = sorted(os.listdir(image_dir))

    volumes = []
    masks = []

    for file in image_files:

        if not file.endswith(".nii") and not file.endswith(".nii.gz"):
            continue

        image_path = os.path.join(image_dir, file)

        mask_file = file.replace("volume", "segmentation")
        mask_path = os.path.join(mask_dir, mask_file)

        if not os.path.exists(mask_path):
            print("Mask not found:", file)
            continue

        image, mask = preprocess_image(image_path, mask_path)

        if image is None:
            continue

        volumes.append(image)
        masks.append(mask)

        print(f"Loaded: {file} | Shape: {image.shape}")

    return volumes, masks

# ===============================
# RUN
# ===============================

image_dir = r"C:\MiniProject\Mini Project\LiTS_small\volumes"
mask_dir  = r"C:\MiniProject\Mini Project\LiTS_small\segmentations"

volumes, masks = load_dataset(image_dir, mask_dir)

print("\nTotal volumes loaded:", len(volumes))

Loaded: volume-0.nii | Shape: (29, 512, 512)
Loaded: volume-1.nii | Shape: (29, 512, 512)
Loaded: volume-10.nii | Shape: (181, 512, 512)
Loaded: volume-100.nii | Shape: (276, 512, 512)
Loaded: volume-101.nii | Shape: (259, 512, 512)
Loaded: volume-102.nii | Shape: (266, 512, 512)
Loaded: volume-103.nii | Shape: (214, 512, 512)
Loaded: volume-104.nii | Shape: (194, 512, 512)
Loaded: volume-105.nii | Shape: (239, 512, 512)
Loaded: volume-106.nii | Shape: (168, 512, 512)

Total volumes loaded: 10


In [3]:
from sklearn.model_selection import train_test_split

volume_names = sorted(os.listdir(image_dir))

train_vols, val_vols = train_test_split(
    volume_names,
    test_size=0.2,
    random_state=42
)

print("Train volumes:", train_vols)
print("Val volumes:", val_vols)

Train volumes: ['volume-102.nii', 'volume-0.nii', 'volume-104.nii', 'volume-10.nii', 'volume-106.nii', 'volume-101.nii', 'volume-100.nii', 'volume-103.nii']
Val volumes: ['volume-105.nii', 'volume-1.nii']


In [4]:
def extract_3d_patches(volume, mask,
                       patch_size=(16,128,128),
                       stride=(8,64,64)):

    D, H, W = volume.shape
    d, h, w = patch_size
    sd, sh, sw = stride

    vol_patches = []
    mask_patches = []

    for z in range(0, D - d + 1, sd):
        for y in range(0, H - h + 1, sh):
            for x in range(0, W - w + 1, sw):

                v_patch = volume[z:z+d, y:y+h, x:x+w]
                m_patch = mask[z:z+d, y:y+h, x:x+w]

                if np.sum(m_patch == 2) > 0:   # tumor
                    vol_patches.append(v_patch)
                    mask_patches.append(m_patch)

                elif np.sum(m_patch == 1) > 0:  # liver
                    vol_patches.append(v_patch)
                    mask_patches.append(m_patch)

                elif np.random.rand() < 0.05:   # background
                    vol_patches.append(v_patch)
                    mask_patches.append(m_patch)

    return vol_patches, mask_patches

In [5]:
def create_patches_from_volumes(volume_list, image_dir, mask_dir):

    X = []
    Y = []

    for file in volume_list:

        image_path = os.path.join(image_dir, file)
        mask_file = file.replace("volume", "segmentation")
        mask_path = os.path.join(mask_dir, mask_file)

        image, mask = preprocess_image(image_path, mask_path)

        if image is None:
            continue

        img_patches, mask_patches = extract_3d_patches(image, mask)

        X.extend(img_patches)
        Y.extend(mask_patches)

        print(f"{file} → {len(img_patches)} patches")

    return np.array(X, dtype=np.float32), \
           np.array(Y, dtype=np.uint8)

In [6]:
print("\nCreating TRAIN patches...")
X_train, Y_train = create_patches_from_volumes(
    train_vols, image_dir, mask_dir
)

print("\nCreating VAL patches...")
X_val, Y_val = create_patches_from_volumes(
    val_vols, image_dir, mask_dir
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)


Creating TRAIN patches...
volume-102.nii → 627 patches
volume-0.nii → 54 patches
volume-104.nii → 520 patches
volume-10.nii → 415 patches
volume-106.nii → 409 patches
volume-101.nii → 796 patches
volume-100.nii → 729 patches
volume-103.nii → 436 patches

Creating VAL patches...
volume-105.nii → 548 patches
volume-1.nii → 65 patches
Train shape: (3986, 16, 128, 128)
Val shape: (613, 16, 128, 128)


In [7]:
np.save("X_train.npy", X_train)
np.save("Y_train.npy", Y_train)

np.save("X_val.npy", X_val)
np.save("Y_val.npy", Y_val)

print("Saved successfully.")

Saved successfully.
